# 01 — Collect Human Reviews from Amazon Reviews 2023

**Run once.** Streams the McAuley-Lab Amazon Reviews 2023 dataset directly from HuggingFace, filters to pre-ChatGPT (before 2022-11-30) **English** reviews of 20–400 words, takes a uniform reservoir sample across 8 product categories, and saves the combined CSV to Google Drive.

- Output: `MyDrive/ECS111FinalProject/data/raw/human_reviews.csv` (10,000 rows)
- Per-category checkpoints land in `MyDrive/ECS111FinalProject/data/raw/human_per_cat/` so reruns skip categories already done.
- Non-English reviews are dropped via `langdetect` — the dataset contains some Spanish/other-language reviews, and the project is English-only.
- No GPU needed. Streaming is network-bound; expect roughly 30–60 min for the full run.

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ECS111FinalProject'
RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')
CKPT_DIR = os.path.join(RAW_DIR, 'human_per_cat')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Saving to:', RAW_DIR)

In [ ]:
!pip install -q requests pandas tqdm langdetect

## Config

- `PRE_CHATGPT_CUTOFF_MS`: 2022-11-30 00:00:00 UTC. Reviews at or after this are dropped — ChatGPT went public on Nov 30 2022, so anything before is confidently human.
- Sampling 10,000 reviews split evenly across 8 categories (1,250 each).

In [ ]:
PRE_CHATGPT_CUTOFF_MS = 1669766400000

CATEGORIES = [
    'All_Beauty',
    'Books',
    'Electronics',
    'Home_and_Kitchen',
    'Sports_and_Outdoors',
    'Toys_and_Games',
    'Pet_Supplies',
    'Office_Products',
]

MIN_WORDS = 20
MAX_WORDS = 400
N_TOTAL = 10_000
SEED = 42

# The Amazon dataset contains some non-English reviews. We stream a bit extra,
# then drop non-English rows and trim back to target — running langdetect on
# every streamed line would be far too slow on the larger category files.
LANG_OVERSAMPLE = 1.15

HF_BASE = (
    'https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/'
    'resolve/main/raw/review_categories'
)

## Streaming reservoir sampler

Each category's JSONL is gigabytes — too big to download fully. We stream line-by-line and use reservoir sampling to keep a uniform random sample of size `target_n` without ever holding the whole file in memory.

In [ ]:
import json
import random
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # make langdetect deterministic


def is_english(text: str) -> bool:
    try:
        return detect(text) == 'en'
    except Exception:
        return False  # undetectable / too short -> treat as non-English, drop it


def stream_category(category: str, target_n: int, seed: int) -> list[dict]:
    url = f'{HF_BASE}/{category}.jsonl'
    rng = random.Random(seed)
    reservoir: list[dict] = []
    n_scanned = 0
    n_kept = 0

    with requests.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        total_bytes = int(resp.headers.get('Content-Length') or 0) or None
        pbar = tqdm(total=total_bytes, desc=category, unit='B',
                    unit_scale=True, unit_divisor=1024, dynamic_ncols=True)
        for raw in resp.iter_lines(decode_unicode=False):
            if not raw:
                continue
            pbar.update(len(raw) + 1)
            n_scanned += 1
            if n_scanned % 5000 == 0:
                pbar.set_postfix(scanned=f'{n_scanned:,}', kept=f'{n_kept:,}')

            try:
                ex = json.loads(raw)
            except json.JSONDecodeError:
                continue

            ts = ex.get('timestamp')
            text = ex.get('text') or ''
            if not ts or ts >= PRE_CHATGPT_CUTOFF_MS:
                continue
            wc = len(text.split())
            if wc < MIN_WORDS or wc > MAX_WORDS:
                continue

            row = {
                'review_id': f"{category}_{ex.get('user_id', '')}_{ts}",
                'category': category,
                'parent_asin': ex.get('parent_asin'),
                'rating': ex.get('rating'),
                'title': ex.get('title'),
                'text': text,
                'timestamp': ts,
                'label': 0,
                'generator': 'human',
            }

            if len(reservoir) < target_n:
                reservoir.append(row)
            else:
                j = rng.randrange(n_kept + 1)
                if j < target_n:
                    reservoir[j] = row
            n_kept += 1

        pbar.set_postfix(scanned=f'{n_scanned:,}', kept=f'{n_kept:,}')
        pbar.close()
        print(f'  {category}: scanned {n_scanned:,}, passed filters {n_kept:,}, sampled {len(reservoir):,}')
    return reservoir

## Run — stream each category with checkpoints

If a category's checkpoint CSV is already on Drive, it's skipped. To force a re-pull, delete that file in Drive and re-run.

In [ ]:
per_cat = N_TOTAL // len(CATEGORIES)
extra = N_TOTAL - per_cat * len(CATEGORIES)
ckpt_dir = Path(CKPT_DIR)

for i, cat in enumerate(CATEGORIES):
    ckpt_path = ckpt_dir / f'{cat}.csv'
    if ckpt_path.exists():
        print(f'  {cat}: checkpoint exists, skipping')
        continue
    target = per_cat + (1 if i < extra else 0)

    # stream extra, drop non-English, then trim back to target
    rows = stream_category(cat, int(target * LANG_OVERSAMPLE), SEED + i)
    en_rows = [r for r in rows if is_english(r['text'])]
    dropped = len(rows) - len(en_rows)
    random.Random(SEED + i).shuffle(en_rows)
    en_rows = en_rows[:target]

    pd.DataFrame(en_rows).to_csv(ckpt_path, index=False)
    print(f'  {cat}: dropped {dropped} non-English, saved {len(en_rows)} rows -> {ckpt_path}')
    if len(en_rows) < target:
        print(f'    WARNING: only {len(en_rows)} English rows (< target {target}); '
              f'raise LANG_OVERSAMPLE and delete this checkpoint to retry')

## Combine, dedupe, shuffle, save final CSV

In [ ]:
parts = [pd.read_csv(ckpt_dir / f'{c}.csv') for c in CATEGORIES if (ckpt_dir / f'{c}.csv').exists()]
assert parts, 'no per-category checkpoints found'

df = pd.concat(parts, ignore_index=True)
df = (
    df.drop_duplicates(subset='review_id')
      .sample(frac=1, random_state=SEED)
      .reset_index(drop=True)
)

out_path = Path(RAW_DIR) / 'human_reviews.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df):,} human reviews to {out_path}')

## Quick sanity check

In [ ]:
print('Category counts:')
print(df['category'].value_counts())
wc = df['text'].str.split().str.len()
print(f'\nWord count: min={wc.min()}, max={wc.max()}, mean={wc.mean():.1f}')
df.head(3)